# SentinelQ - Defect Detection Training (YOLOv8, Google Colab)

Trains a **YOLOv8n** object detector on **NEU-DET** (steel surface defects, 6 classes with
bounding boxes) and exports weights for the SentinelQ backend, so the detector draws a
labeled box on each defect (crazing, inclusion, patches, pitted_surface, rolled-in_scale,
scratches).

## How to run
1. Runtime -> Change runtime type -> **GPU**.
2. Runtime -> **Run all**.
3. Upload your `kaggle.json` when prompted (Kaggle -> Account -> *Create New API Token*).
4. At the end, **detection_v1.zip** downloads.
5. Unzip it into the repo's `models/` folder so you have `models/detection/v1/weights.pt`.
6. From `backend/`, run:
   `python -m scripts.register_weights --model-type detection --version v1 --activate`

The backend loads this `weights.pt` with ultralytics `YOLO(...)`, so it plugs straight in.


In [ ]:
!pip -q install ultralytics kagglehub

In [ ]:
import os
from google.colab import files

print('Upload your kaggle.json (Kaggle > Account > Create New API Token)')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for name in list(uploaded):
    if name.endswith('.json'):
        os.replace(name, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed')

In [ ]:
import kagglehub

# NEU-DET with bounding-box annotations. If this slug is unavailable, replace it with
# any Kaggle NEU-DET (object detection) dataset slug.
KAGGLE_DATASET = 'kaustubhdikshit/neu-surface-defect-database'

path = kagglehub.dataset_download(KAGGLE_DATASET)
print('downloaded to', path)

In [ ]:
import glob
import os
import random
import shutil
import xml.etree.ElementTree as ET

random.seed(42)

# Canonical NEU-DET classes; normalize common spelling variants to these.
CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
ALIASES = {
    'crazing': 'crazing', 'cr': 'crazing',
    'inclusion': 'inclusion', 'in': 'inclusion',
    'patches': 'patches', 'pa': 'patches',
    'pitted_surface': 'pitted_surface', 'pitted-surface': 'pitted_surface', 'ps': 'pitted_surface',
    'rolled-in_scale': 'rolled-in_scale', 'rolled_in_scale': 'rolled-in_scale',
    'rolled-in-scale': 'rolled-in_scale', 'rs': 'rolled-in_scale',
    'scratches': 'scratches', 'sc': 'scratches',
}
CLASS_INDEX = {name: i for i, name in enumerate(CLASSES)}


def canonical(name):
    key = name.strip().lower().replace(' ', '_')
    return ALIASES.get(key, key)


xml_files = glob.glob(os.path.join(path, '**', '*.xml'), recursive=True)
images = {}
for ext in ('jpg', 'jpeg', 'png', 'bmp'):
    for img in glob.glob(os.path.join(path, '**', f'*.{ext}'), recursive=True):
        images.setdefault(os.path.splitext(os.path.basename(img))[0], img)
print('found', len(xml_files), 'annotations and', len(images), 'images')

ROOT = '/content/neu'
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)

random.shuffle(xml_files)
n_val = max(1, int(0.15 * len(xml_files)))
val_set = set(xml_files[:n_val])
unknown = set()
kept = 0

for xml in xml_files:
    stem = os.path.splitext(os.path.basename(xml))[0]
    img_path = images.get(stem)
    if img_path is None:
        continue
    tree = ET.parse(xml)
    root = tree.getroot()
    size = root.find('size')
    w = float(size.find('width').text)
    h = float(size.find('height').text)
    lines = []
    for obj in root.findall('object'):
        name = canonical(obj.find('name').text)
        if name not in CLASS_INDEX:
            unknown.add(name)
            continue
        b = obj.find('bndbox')
        x1, y1 = float(b.find('xmin').text), float(b.find('ymin').text)
        x2, y2 = float(b.find('xmax').text), float(b.find('ymax').text)
        cx, cy = ((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h
        bw, bh = (x2 - x1) / w, (y2 - y1) / h
        lines.append(f'{CLASS_INDEX[name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    if not lines:
        continue
    split = 'val' if xml in val_set else 'train'
    shutil.copy(img_path, os.path.join(ROOT, 'images', split, os.path.basename(img_path)))
    with open(os.path.join(ROOT, 'labels', split, stem + '.txt'), 'w') as fh:
        fh.write('\n'.join(lines))
    kept += 1

print('converted', kept, 'labeled images. unknown classes seen:', sorted(unknown))

with open('/content/neu.yaml', 'w') as fh:
    fh.write('path: /content/neu\n')
    fh.write('train: images/train\n')
    fh.write('val: images/val\n')
    fh.write(f'names: {CLASSES}\n')
print(open('/content/neu.yaml').read())

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(data='/content/neu.yaml', epochs=60, imgsz=320, batch=32, patience=15, seed=42)
metrics = model.val()
print('mAP50    =', round(float(metrics.box.map50), 4))
print('mAP50-95 =', round(float(metrics.box.map), 4))

In [ ]:
import glob
import json
import os
import shutil

best = sorted(glob.glob('runs/detect/**/weights/best.pt', recursive=True), key=os.path.getmtime)[-1]
os.makedirs('detection/v1', exist_ok=True)
shutil.copy(best, 'detection/v1/weights.pt')

json.dump({
    'model_type': 'detection', 'version': 'v1', 'arch': 'yolov8n',
    'classes': CLASSES, 'imgsz': 320, 'epochs': 60, 'dataset': KAGGLE_DATASET, 'seed': 42,
}, open('detection/v1/config.json', 'w'), indent=2)

json.dump({
    'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
    'classes': CLASSES,
}, open('detection/v1/metrics.json', 'w'), indent=2)

# Archive with the 'detection/' folder at the zip root so it unzips to models/detection/v1/.
shutil.make_archive('detection_v1', 'zip', root_dir='.', base_dir='detection')
from google.colab import files
files.download('detection_v1.zip')
print('Done. Unzip detection_v1.zip into the repo models/ folder -> models/detection/v1/*')